# Conteneurisation de l'API

L'API est construite en image Docker, publiée dans le registre du projet, puis relancée au démarrage de la VM par systemd.
Prérequis cloud : VM `docker` avec accès en lecture à Cloud Source, puis accès au stockage pour le push. Bucket et VM MLflow déjà en place.
Passer `ENV`, `MLFLOW_SERVER` et `MLFLOW_REGISTRY_NAME` au conteneur. Sans ces variables, le chargement de `src` s'arrête. Ne pas lancer Gunicorn en daemon dans le `CMD` : le conteneur se terminerait.
Le nom d'hôte interne de la VM MLflow n'est résolu que dans le projet GCP. Le port publié est 80.


## Image

Créer la VM `docker`. Niveau d'accès : définir l'accès pour chaque API, Cloud Source Repositories en lecture seule.
Cloner avec le SDK, pas en SSH : aucune clé Git n'est configurée sur cette VM. Se placer sur `staging`.


Dockerfile du projet. `libgomp1` est requis par LightGBM. Le `CMD` reste au premier plan.


Ce premier `docker run` échoue : les variables d'environnement ne sont pas définies.
Les placer dans `env.list` et les passer avec `--env-file`.
Sur une VM du même projet, le compte de service par défaut lit déjà Cloud Storage. Hors de ce projet, il faudrait une clé et `GOOGLE_APPLICATION_CREDENTIALS` dans ce fichier.


## Appel


In [ ]:
import os
import requests
import pandas as pd

dataset = pd.read_csv(os.path.expanduser("data/primary.csv"))
dataset = dataset.drop(["user_session", "user_id", "purchased"], axis=1)


In [ ]:
requests.post(
    "http://<IP_INSTANCE>/predict",  # Remplacer par l'adresse IP de l'instance Docker
    json=dataset.sample(n=10).to_json()
).json()


## Registre et démarrage automatique

Héberger l'image dans Artifact Registry, privé au projet. Format Docker. Nom de dépôt du TP : `purchased-docker` (le caractère `_` est refusé). Scan de vulnérabilités désactivé pour le TP.
Arrêter la VM et lui ajouter l'accès API au stockage avant le push. `sudo` écrit la configuration Docker dans `/root`.
Les commandes de tag et de push reprennent la référence du cours.


## Nom d'hôte MLflow

Sur la VM MLflow, `hostname -f` donne le nom interne. Il reste valable si l'adresse IP publique change.
Depuis la VM Docker, un ping vérifie la résolution. Depuis un poste hors du projet, le nom n'est pas résolu.


## Service systemd

Les variables du service sont dans `/etc/default/purchase_predict_api`, pas via `export`.
`ExecStartPre` avec `-` ignore l'échec de `stop` et `rm` si le conteneur n'existe pas encore.


Fichier `/etc/systemd/system/purchase_predict_api.service`.


Le port 80 doit être en écoute. Sinon, lire le journal du service.


In [ ]:
requests.post(
    "http://<IP_INSTANCE>/predict",  # Remplacer par l'adresse IP de l'instance Docker
    json=dataset.sample(n=10).to_json()
).json()


Après redémarrage de la VM, le service relance le conteneur. L'adresse IP publique est éphémère : mettre à jour l'IP de la requête si elle a changé.
Arrêter l'instance une fois les vérifications terminées.
